In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

housing = fetch_california_housing()

df = pd.DataFrame(housing.data, columns=housing.feature_names)

target = pd.Series(housing.target, name="MedHouseVal")

df = pd.concat([df, target], axis=1)

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (16512, 8)
Testing data: (4128, 8)


In [6]:
model = LinearRegression()

rfe = RFE(
    estimator=model,
    n_features_to_select=5
)

rfe.fit(X_train, y_train)

selected_features = X.columns[rfe.support_]

print("Selected features:")
print(selected_features.tolist())

print("\nFeature ranking:")
print(pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rfe.ranking_
}))

Selected features:
['MedInc', 'AveRooms', 'AveBedrms', 'Latitude', 'Longitude']

Feature ranking:
      Feature  Ranking
0      MedInc        1
1    HouseAge        2
2    AveRooms        1
3   AveBedrms        1
4  Population        4
5    AveOccup        3
6    Latitude        1
7   Longitude        1


In [5]:
model_all = LinearRegression()

model_all.fit(X_train, y_train)

pred_all = model_all.predict(X_test)

mae_all = mean_absolute_error(y_test, pred_all)
mse_all = mean_squared_error(y_test, pred_all)
rmse_all = np.sqrt(mse_all)
r2_all = r2_score(y_test, pred_all)

In [7]:
model_selected = LinearRegression()

model_selected.fit(
    X_train[selected_features],
    y_train
)

pred_selected = model_selected.predict(
    X_test[selected_features]
)

mae_selected = mean_absolute_error(y_test, pred_selected)
mse_selected = mean_squared_error(y_test, pred_selected)
rmse_selected = np.sqrt(mse_selected)
r2_selected = r2_score(y_test, pred_selected)

In [8]:
results = pd.DataFrame({
    "Model": [
        "All Features",
        "RFE Selected Features"
    ],
    "Number of Features": [
        8,
        len(selected_features)
    ],
    "MAE": [
        mae_all,
        mae_selected
    ],
    "MSE": [
        mse_all,
        mse_selected
    ],
    "RMSE": [
        rmse_all,
        rmse_selected
    ],
    "R2 Score": [
        r2_all,
        r2_selected
    ]
})

results.round(4)

,Model,Number of Features,MAE,MSE,RMSE,R2 Score
0,All Features,8,0.5332,0.5559,0.7456,0.5758
1,RFE Selected Features,5,0.5393,0.5668,0.7528,0.5675


In [9]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_all.coef_
})

importance["Absolute Coefficient"] = importance["Coefficient"].abs()

importance = importance.sort_values(
    "Absolute Coefficient",
    ascending=False
)

importance

,Feature,Coefficient,Absolute Coefficient
3,AveBedrms,0.783145,0.783145
0,MedInc,0.448675,0.448675
7,Longitude,-0.433708,0.433708
6,Latitude,-0.419792,0.419792
2,AveRooms,-0.123323,0.123323
1,HouseAge,0.009724,0.009724
5,AveOccup,-0.003526,0.003526
4,Population,-0.000002,0.000002


In [10]:
print("Selected features using RFE:")
print(selected_features.tolist())

print("\nModel comparison:")
print(results.round(4))

print("\nConclusion:")

if r2_selected > r2_all:
    print("The RFE model has a higher R2 Score than the model using all features.")
    print("Therefore, reducing the number of features improved prediction accuracy.")
elif r2_selected < r2_all:
    print("The model using all features has a higher R2 Score than the RFE model.")
    print("Therefore, reducing the number of features decreased prediction accuracy.")
else:
    print("Both models have the same R2 Score.")
    print("Therefore, reducing the number of features maintained prediction accuracy.")

print("\nSelected features are:")
print(selected_features.tolist())

Selected features using RFE:
['MedInc', 'AveRooms', 'AveBedrms', 'Latitude', 'Longitude']

Model comparison:
                   Model  Number of Features     MAE     MSE    RMSE  R2 Score
0           All Features                   8  0.5332  0.5559  0.7456    0.5758
1  RFE Selected Features                   5  0.5393  0.5668  0.7528    0.5675

Conclusion:
The model using all features has a higher R2 Score than the RFE model.
Therefore, reducing the number of features decreased prediction accuracy.

Selected features are:
['MedInc', 'AveRooms', 'AveBedrms', 'Latitude', 'Longitude']
